# Small World: construindo (e atravessando) uma rede de páginas da Wikipédia

> **Provocação inicial.** Em 1967, Stanley Milgram entregou cartas a pessoas em Omaha (Nebraska) e pediu que elas chegassem a um corretor em Boston, passando apenas por conhecidos. As cartas que chegaram precisaram, em média, de **seis** intermediários. Matemáticos medem sua distância até Paul Erdős (o *número de Erdős*); cinéfilos medem a distância até Kevin Bacon (o *Oracle of Bacon*); e no [WikiGame](https://www.thewikigame.com) você tenta ir de *Walmart* a *Hebrew language* só clicando em links.
>
> **Todos esses jogos são o mesmo problema matemático**: encontrar caminhos curtos em um grafo gigante que ninguém desenhou de propósito.

Nesta aula você vai **construir o seu próprio WikiGame**. Em vez de jogar, você vai:

1. **Coletar** automaticamente uma rede com milhares de páginas da Wikipédia, a partir de uma única semente;
2. **Entender de verdade** a diferença entre busca em largura (BFS) e em profundidade (DFS), implementando as duas e vendo-as funcionar quadro a quadro;
3. **Limpar e podar** a rede (dados reais são sujos);
4. **Atravessar** a rede: menores caminhos, "número de Erdős" de cada página, diâmetro, e o teste decisivo: *essa rede é um mundo pequeno?*

Ao final, você terá um notebook que funciona como um laboratório: troque a semente e repita a análise com qualquer assunto da Wikipédia.

**Como usar este notebook:** leia os blocos **🧠 Pare e pense** antes de rodar a célula seguinte. Eles existem para você apostar em uma resposta *antes* de ver o resultado. Quem aposta lembra; quem só lê esquece.

## 1. Três formas de construir uma rede

Até aqui vimos duas maneiras de criar um grafo em NetworkX: a partir de um **arquivo CSV** e **manualmente**, nó por nó. Para redes pequenas isso funciona. Para redes médias ou grandes, ou quando a análise precisa ser repetida muitas vezes, é inviável.

A terceira forma é **coletar nós e arestas automaticamente da Internet**. A Wikipédia é perfeita para isso:

* cada **página** é um nó;
* cada **link** de uma página para outra é uma aresta **dirigida** (se A aponta para B, nada garante que B aponte para A).

Começamos pela página [Complex network](https://en.wikipedia.org/wiki/Complex_network), a **semente**. As páginas que ela cita são, presumivelmente, relacionadas (senão, por que os editores as citariam?). Depois visitamos as páginas citadas, e as citadas pelas citadas, como uma bola de neve rolando morro abaixo. Esse processo se chama **amostragem snowball**, e o algoritmo por trás dele é a **busca em largura (BFS)**.

A rede resultante responde a uma pergunta bem concreta: *onde a análise de redes complexas se encaixa no mapa das outras disciplinas?*

In [ ]:
# Instalação (só a primeira vez no Colab)
!pip install -q wikipedia plotly

In [ ]:
from collections import deque
from operator import itemgetter
import random
import warnings

import networkx as nx
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import wikipedia

warnings.filterwarnings("ignore")  # silencia o aviso do parser HTML da lib wikipedia

# Paleta usada em todo o notebook (cores fixas por papel, nunca por posição)
AZUL, LARANJA, VERDE, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#b8b7b0"

print("NetworkX", nx.__version__)

## 2. Antes da Wikipédia: BFS e DFS com as próprias mãos

Os slides mostram um grafo de 10 nós e duas ordens de visita. Vamos reproduzi-lo aqui e **implementar os dois algoritmos do zero**, sem NetworkX, porque a diferença entre eles cabe em uma única linha de código.

Os dois algoritmos mantêm uma **fronteira**: o conjunto de nós já descobertos mas ainda não visitados.

* **BFS** usa uma **fila** (FIFO): quem foi descoberto primeiro é visitado primeiro. Resultado: a busca se expande em **camadas**, como ondas em um lago.
* **DFS** usa uma **pilha** (LIFO): quem foi descoberto por último é visitado primeiro. Resultado: a busca **mergulha** por um caminho até o fim antes de voltar.

🧠 **Pare e pense.** No grafo abaixo, partindo do nó 1, qual será o **quarto** nó visitado pela BFS? E pela DFS? Anote antes de rodar.

In [ ]:
# O grafo dos slides
toy = nx.Graph()
toy.add_edges_from([(1, 2), (1, 3), (2, 4), (2, 5), (2, 6), (5, 6),
                    (3, 7), (3, 8), (4, 9), (4, 10)])

def bfs(G, origem):
    '''Retorna a ordem de visita e o tamanho da fronteira a cada passo.'''
    fronteira = deque([origem])           # FILA
    visitados, ordem, tamanhos = {origem}, [], []
    while fronteira:
        tamanhos.append(len(fronteira))
        no = fronteira.popleft()          # tira do INÍCIO
        ordem.append(no)
        for viz in sorted(G[no]):
            if viz not in visitados:
                visitados.add(viz)
                fronteira.append(viz)
    return ordem, tamanhos

def dfs(G, origem):
    fronteira = [origem]                  # PILHA
    visitados, ordem, tamanhos = set(), [], []
    while fronteira:
        tamanhos.append(len(fronteira))
        no = fronteira.pop()              # tira do FIM
        if no in visitados:
            continue
        visitados.add(no)
        ordem.append(no)
        for viz in sorted(G[no], reverse=True):   # reverse p/ visitar o menor primeiro
            if viz not in visitados:
                fronteira.append(viz)
    return ordem, tamanhos

ordem_bfs, front_bfs = bfs(toy, 1)
ordem_dfs, front_dfs = dfs(toy, 1)
print("BFS:", ordem_bfs)
print("DFS:", ordem_dfs)

Repare: **as duas funções são idênticas exceto por `popleft()` versus `pop()`**. Fila ou pilha. É só isso que separa "explorar em ondas" de "mergulhar fundo".

A animação abaixo mostra os dois algoritmos avançando passo a passo. Use o controle deslizante ou o botão ▶.

In [ ]:
pos_toy = {1: (0, 3), 2: (-1.5, 2), 3: (1.5, 2), 4: (-2.5, 1), 5: (-1.5, 1), 6: (-0.5, 1),
           7: (1, 1), 8: (2, 1), 9: (-3, 0), 10: (-2, 0)}

def quadro(G, pos, ordem, passo, cor):
    '''Desenha o grafo com os 'passo' primeiros nós da ordem destacados.'''
    visitados = ordem[:passo]
    ex, ey = [], []
    for u, v in G.edges():
        ex += [pos[u][0], pos[v][0], None]; ey += [pos[u][1], pos[v][1], None]
    arestas = go.Scatter(x=ex, y=ey, mode="lines", line=dict(color=CINZA, width=2), hoverinfo="skip")
    nos = go.Scatter(
        x=[pos[n][0] for n in G], y=[pos[n][1] for n in G], mode="markers+text",
        text=[f"{n}<br><b>{visitados.index(n)+1}º</b>" if n in visitados else str(n) for n in G],
        textposition="middle center", textfont=dict(color="white", size=11),
        marker=dict(size=44, color=[cor if n in visitados else CINZA for n in G],
                    line=dict(color="white", width=2)),
        hovertext=[f"nó {n}" for n in G], hoverinfo="text")
    return [arestas, nos]

def animar(G, pos, ordem, titulo, cor):
    frames = [go.Frame(data=quadro(G, pos, ordem, k, cor), name=str(k)) for k in range(len(ordem) + 1)]
    fig = go.Figure(data=frames[0].data, frames=frames)
    fig.update_layout(
        title=titulo, showlegend=False, width=650, height=420, plot_bgcolor="white",
        xaxis=dict(visible=False), yaxis=dict(visible=False), margin=dict(l=10, r=10, t=50, b=10),
        updatemenus=[dict(type="buttons", showactive=False, y=1.12, x=1.0, xanchor="right",
            buttons=[dict(label="▶ Rodar", method="animate",
                          args=[None, dict(frame=dict(duration=700, redraw=True), fromcurrent=True)])])],
        sliders=[dict(steps=[dict(method="animate", label=str(k),
                                  args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                             for k in range(len(ordem) + 1)],
                      currentvalue=dict(prefix="passo: "))])
    return fig

animar(toy, pos_toy, ordem_bfs, "BFS: fila (FIFO) — a busca avança em camadas", AZUL).show()
animar(toy, pos_toy, ordem_dfs, "DFS: pilha (LIFO) — a busca mergulha e volta", LARANJA).show()

### 2.1 O preço de cada estratégia: o tamanho da fronteira

A tabela dos slides diz que a BFS tem **risco de explosão de memória** e a DFS tem **risco de caminhos improdutivos**. Vamos ver o primeiro acontecer. O gráfico abaixo mostra quantos nós estavam na fronteira em cada passo. Pense na fronteira como a "lista de pendências" do algoritmo.

In [ ]:
fig = go.Figure()
fig.add_scatter(y=front_bfs, mode="lines+markers", name="BFS (fila)", line=dict(color=AZUL, width=2), marker=dict(size=9))
fig.add_scatter(y=front_dfs, mode="lines+markers", name="DFS (pilha)", line=dict(color=LARANJA, width=2), marker=dict(size=9))
fig.update_layout(title="Tamanho da fronteira a cada passo (grafo de 10 nós)",
                  xaxis_title="passo", yaxis_title="nós na fronteira", width=650, height=380,
                  plot_bgcolor="white", legend=dict(orientation="h", y=1.1))
fig.update_yaxes(gridcolor="#eee")
fig.show()

Em um grafo de 10 nós a diferença é pequena. Agora imagine um nó com 100 vizinhos, cada um com 100 vizinhos. A BFS precisa guardar **toda a camada** antes de avançar: $100$, depois $10^4$, depois $10^6$ nós na fronteira. A DFS guarda só o caminho atual.

🧠 **Pare e pense.** Se a DFS gasta tão menos memória, por que a coleta da Wikipédia que faremos a seguir usa BFS? *(Dica: olhe a linha "Goal guarantee" da tabela dos slides. E pergunte-se: na Wikipédia, o que fica a 3 cliques de "Complex network"? E a 30 cliques?)*

### 2.2 NetworkX já sabe fazer isso

Implementamos na mão para entender. No dia a dia, use as versões prontas, que são testadas e otimizadas:

In [ ]:
print("BFS (nx):   ", list(nx.bfs_tree(toy, 1)))
print("DFS (nx):   ", list(nx.dfs_preorder_nodes(toy, 1)))
print("Camadas BFS:", list(nx.bfs_layers(toy, 1)))   # exatamente as 'ondas' que vimos na animação

## 3. A bola de neve na Wikipédia

Agora o grafo real. Precisamos de duas decisões antes de começar.

**Decisão 1: a semente.** `SEED` é o nome da página inicial. Trocando essa constante você aplica toda a análise a qualquer outro assunto.

**Decisão 2: a lista negra.** Quase toda página da Wikipédia cita *ISBN*, *DOI*, *PubMed*, *arXiv* etc. Se deixarmos esses links entrarem, **qualquer** rede da Wikipédia vira um punhado de estrelas gigantes idênticas: a *síndrome do estrelato*. Tratamos essas páginas como *stop words* em recuperação de informação e simplesmente as ignoramos. A lista `STOPS` foi montada por tentativa e erro; você vai ampliá-la quando encontrar novas estrelas. Também ignoramos páginas que começam com *"List Of"*, que são só índices de outros assuntos.

In [ ]:
SEED = "Complex network".title()

STOPS = ("International Standard Serial Number", "International Standard Book Number",
         "National Diet Library", "International Standard Name Identifier",
         "International Standard Book Number (Identifier)", "Pubmed Identifier",
         "Pubmed Central", "Digital Object Identifier", "Arxiv", "Proc Natl Acad Sci Usa",
         "Bibcode", "Library Of Congress Control Number", "Jstor", "Doi (Identifier)",
         "Isbn (Identifier)", "Pmid (Identifier)", "Arxiv (Identifier)", "Bibcode (Identifier)",
         "S2Cid (Identifier)", "Issn (Identifier)", "Pmc (Identifier)", "Hdl (Identifier)",
         "Citeseerx (Identifier)", "Jstor (Identifier)", "Oclc (Identifier)", "Wayback Machine")

### 3.1 Quantas camadas podemos coletar?

A Wikipédia em inglês tem mais de 7 milhões de artigos. Suponha que cada página tenha $N \approx 100$ links. A camada 1 tem $N$ páginas, a camada 2 tem $N^2$, a camada 3 tem $N^3$... Para *processar* a camada $k$ inteira precisamos baixar todas as suas páginas.

🧠 **Pare e pense.** Baixando uma página por segundo, quanto tempo leva para coletar até a camada 3? Chute antes de rodar.

In [ ]:
N = 100
camadas = np.arange(0, 5)
paginas = N ** camadas
segundos = np.cumsum(paginas)

fig = go.Figure(go.Bar(x=[f"camada {k}" for k in camadas], y=paginas, marker_color=AZUL,
                       text=[f"{p:,} págs ≈ {s/3600:.1f} h" if s > 3600 else f"{p:,} págs ≈ {s} s"
                             for p, s in zip(paginas, segundos)], textposition="outside"))
fig.update_layout(title="Explosão combinatória: páginas a baixar por camada (N = 100)",
                  yaxis_type="log", yaxis_title="páginas (escala log)", width=650, height=400,
                  plot_bgcolor="white")
fig.update_yaxes(gridcolor="#eee")
fig.show()
print(f"Até a camada 3: {segundos[3]/86400:.0f} dias. Até a camada 4: {segundos[4]/86400/365:.0f} anos.")

Por isso vamos **processar** apenas a semente (camada 0) e seus vizinhos imediatos (camada 1). As páginas da camada 2 entram no grafo como nós (porque a camada 1 aponta para elas), mas **não são baixadas**. Guarde esse detalhe: ele explica várias coisas estranhas que veremos na rede.

### 3.2 As estruturas de dados da BFS

Uma BFS precisa lembrar quais páginas **já foram processadas** e quais foram **descobertas mas ainda não processadas**:

* `done_set`: páginas processadas (um `set`, busca instantânea);
* `todo`: a fila de pendências, em ordem (um `deque`, como na Seção 2);
* `todo_set`: as mesmas páginas da fila, mas em um `set`, para perguntar "já está na fila?" sem percorrer a fila inteira.

Cada item da fila é uma tupla `(camada, página)`, para sabermos a distância até a semente.


In [ ]:
todo = deque([(0, SEED)])   # a semente está na camada 0
todo_set = {SEED}
done_set = set()

g = nx.DiGraph()            # dirigido: link de A para B não implica link de B para A
g.add_node(SEED, layer=0)

### 3.3 O laço principal

Esta célula é a Seção 2 de novo, só que cada "vizinho" custa uma requisição HTTP. Leva entre 1 e 3 minutos. Enquanto roda, acompanhe o progresso e repare quais páginas falham ao carregar (páginas de desambiguação, redirecionamentos quebrados, etc.).

In [ ]:
%%time
MAX_LAYER = 1    # processar só camadas 0 e 1. Mude para 2 se tiver paciência (e uma tarde livre).
falhas = []

while todo:
    layer, page = todo[0]
    if layer > MAX_LAYER:
        break
    todo.popleft()
    done_set.add(page)
    print(layer, page, end=" | ")

    try:
        wiki = wikipedia.page(page, auto_suggest=False)
    except Exception:
        falhas.append(page)
        continue

    for link in wiki.links:
        link = link.title()
        if link in STOPS or link.startswith("List Of"):
            continue
        if link not in todo_set and link not in done_set:
            todo.append((layer + 1, link))
            todo_set.add(link)
            g.add_node(link, layer=layer + 1)
        g.add_edge(page, link)

print(f"\n\n{len(g):,} nós, {g.number_of_edges():,} arestas. Páginas que não carregaram: {len(falhas)}")

A rede está na variável `g`. Mas está **suja**: incompleta, com duplicatas e com um viés de amostragem que precisamos entender antes de tirar qualquer conclusão.

🧠 **Pare e pense.** Quantos nós estão em cada camada? Antes de rodar, estime a razão entre a camada 2 e a camada 1.

In [ ]:
original = g.copy()   # guardamos a versão crua para comparação

camada = nx.get_node_attributes(g, "layer")
contagem = {k: sum(1 for v in camada.values() if v == k) for k in sorted(set(camada.values()))}
fig = go.Figure(go.Bar(x=[f"camada {k}" for k in contagem], y=list(contagem.values()),
                       marker_color=AZUL, text=list(contagem.values()), textposition="outside"))
fig.update_layout(title="Nós por camada (distância BFS até a semente)", width=500, height=380,
                  plot_bgcolor="white", yaxis_title="nós")
fig.show()

## 4. Limpeza: eliminando duplicatas

Muitas páginas existem sob dois ou mais nomes. *Complex Network* e *Complex Networks* são a mesma página (a segunda redireciona para a primeira), mas o NetworkX não sabe disso. Resolver isso de forma completa exigiria ferramentas de processamento de linguagem natural, fora do escopo do curso. Vamos atacar só os dois casos mais comuns: o **plural em "s"** e o **hífen no meio**.

A ferramenta é `nx.contracted_nodes(g, u, v)`, que **funde** o nó `v` no nó `u`: todas as arestas que chegavam ou saíam de `v` passam a chegar ou sair de `u`. Dois detalhes:

* se havia uma aresta `v → u`, a contração a transformaria em um *self-loop* (`u → u`); `self_loops=False` evita isso;
* removemos os self-loops já existentes *antes*, porque eles atrapalham a detecção de duplicatas.

🧠 **Pare e pense.** Por que fundimos "Networks" *em* "Network" e não o contrário? Importa?

In [ ]:
g.remove_edges_from(nx.selfloop_edges(g))

# Caso 1: plural. ('Complex Network', 'Complex Networks')
dup_s = [(n, n + "s") for n in g if n + "s" in g]
for u, v in dup_s:
    g = nx.contracted_nodes(g, u, v, self_loops=False)

# Caso 2: hífen. ('Small-World Network', 'Small World Network')
dup_h = [(n, n.replace("-", " ")) for n in g if "-" in n and n.replace("-", " ") in g]
for u, v in dup_h:
    g = nx.contracted_nodes(g, u, v, self_loops=False)

# contracted_nodes cria um atributo 'contraction' com um dicionário dentro;
# GraphML não aceita dicionários, então zeramos o atributo para poder exportar depois.
nx.set_node_attributes(g, 0, "contraction")
nx.set_edge_attributes(g, 0, "contraction")

print(f"{len(dup_s)} duplicatas por plural, {len(dup_h)} por hífen. Exemplos:")
print("  ", dup_s[:5])
print("  ", dup_h[:5])
print(f"\nAntes: {len(original):,} nós, {original.number_of_edges():,} arestas")
print(f"Depois: {len(g):,} nós, {g.number_of_edges():,} arestas")

## 5. Poda: separando o núcleo da franja

Por que passamos por todo esse trabalho? Primeiro, para construir a rede de assuntos ligados a redes complexas. Segundo, para descobrir **quais assuntos são mais significativos**. Mas o que mede significância?

Ao longo do curso você verá várias medidas. Por enquanto, a mais simples: o **grau de entrada** (*indegree*), o número de links que **apontam para** a página. Se muitas páginas citam um assunto, ele deve ser importante. (O número de links que **saem** da página é o *outdegree*.)

🧠 **Pare e pense.** Antes de olhar a distribuição de grau, responda: que fração dos nós você acha que tem grau 1? Lembre-se de *como* a rede foi coletada (camada 2 não foi baixada).

In [ ]:
graus = np.array([d for _, d in g.degree()])
valores, freq = np.unique(graus, return_counts=True)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Quantos nós têm cada grau (escala linear)", "A mesma coisa em escala log-log"))
fig.add_bar(x=valores[:30], y=freq[:30], marker_color=AZUL, name="nós", row=1, col=1)
fig.add_scatter(x=valores, y=freq, mode="markers", marker=dict(color=AZUL, size=7), name="nós", row=1, col=2)
fig.update_xaxes(title_text="grau", row=1, col=1)
fig.update_xaxes(title_text="grau (log)", type="log", row=1, col=2)
fig.update_yaxes(title_text="nós", row=1, col=1, gridcolor="#eee")
fig.update_yaxes(title_text="nós (log)", type="log", row=1, col=2, gridcolor="#eee")
fig.update_layout(showlegend=False, width=900, height=400, plot_bgcolor="white",
                  title="Distribuição de grau da rede crua")
fig.show()

print(f"Grau médio: {graus.mean():.2f}")
print(f"Nós com grau 1: {100*(graus == 1).mean():.1f}%")
print(f"Nós isolados (grau 0): {(graus == 0).sum()}   <- por que é impossível haver algum?")

A maioria esmagadora dos nós tem **uma única conexão**: são páginas da camada 2, citadas por exatamente uma página da camada 1 e que nunca foram baixadas. Elas são a *franja* da bola de neve, não o *núcleo*. O gráfico log-log mostra uma cauda longa: poucos nós com grau altíssimo, muitos com grau baixo. Guarde essa imagem, ela volta na aula sobre redes livres de escala.

Podemos remover todos os nós de grau 1 para tornar a rede mais compacta e menos "cabeluda" sem prejudicar as conclusões. Chamaremos o que sobra de `gsub`.

In [ ]:
core = [n for n, d in g.degree() if d >= 2]
gsub = nx.DiGraph(nx.subgraph(g, core))     # cópia independente, não uma 'view'

print(f"gsub: {len(gsub):,} nós, {gsub.number_of_edges():,} arestas")
print(f"Nós removidos: {100*(1 - len(gsub)/len(g)):.1f}%")
print(f"Arestas removidas: {100*(1 - gsub.number_of_edges()/g.number_of_edges()):.1f}%")
print(f"Arestas por nó: antes {g.number_of_edges()/len(g):.2f}, depois {gsub.number_of_edges()/len(gsub):.2f}")

nx.write_graphml(gsub, "cna.graphml")   # para abrir no Gephi ou em outra ferramenta

🧠 **Pare e pense.** Removemos cerca de três quartos dos nós, mas menos da metade das arestas. Por que a assimetria? E o que aconteceria com a densidade se repetíssemos a poda pedindo grau $\geq 3$, $\geq 4$, ...? *(Quem quiser ir além: pesquise `nx.k_core`. É exatamente essa ideia, levada até o fim.)*

## 6. Quem manda na rede?

A figura abaixo é equivalente ao que você veria no Gephi, mas **interativa**: passe o mouse sobre os nós, arraste para mover, use o scroll para zoom. Para caber na tela, mostramos só os nós de `gsub` com maior grau de entrada. O tamanho do nó é o indegree; a cor é a camada em que o nó foi descoberto.

In [ ]:
def desenhar_rede(G, pos, tamanho, cor_por, titulo, destaque=None, rotulos=True, largura=900, altura=700,
                  cor_titulo="camada"):
    '''Desenha G com Plotly. tamanho e cor_por são dicionários nó -> valor.
       destaque é uma lista de nós (um caminho) a realçar.'''
    ex, ey = [], []
    for u, v in G.edges():
        ex += [pos[u][0], pos[v][0], None]; ey += [pos[u][1], pos[v][1], None]
    traces = [go.Scatter(x=ex, y=ey, mode="lines", line=dict(color="#d9d8d2", width=0.7),
                         hoverinfo="skip", name="links")]
    if destaque:
        hx, hy = [], []
        for u, v in zip(destaque[:-1], destaque[1:]):
            hx += [pos[u][0], pos[v][0], None]; hy += [pos[u][1], pos[v][1], None]
        traces.append(go.Scatter(x=hx, y=hy, mode="lines", line=dict(color=LARANJA, width=4),
                                 hoverinfo="skip", name="caminho"))
    nos = list(G)
    tam = np.array([tamanho[n] for n in nos], dtype=float)
    tam = 7 + 24 * np.sqrt((tam - tam.min()) / (tam.max() - tam.min() + 1e-9))
    traces.append(go.Scatter(
        x=[pos[n][0] for n in nos], y=[pos[n][1] for n in nos],
        mode="markers+text" if rotulos else "markers",
        text=[n if (rotulos and tamanho[n] >= np.percentile([tamanho[n] for n in nos], 90)) else "" for n in nos],
        textposition="top center", textfont=dict(size=9, color="#52514e"),
        marker=dict(size=tam, color=[cor_por[n] for n in nos],
                    colorscale=[[0, "#9ecae1"], [0.5, "#3182bd"], [1, "#08306b"]],
                    cmin=min(cor_por.values()), cmax=max(cor_por.values()),
                    colorbar=dict(title=cor_titulo, thickness=12), line=dict(color="white", width=1),
                    showscale=True),
        hovertext=[f"<b>{n}</b><br>indegree: {tamanho[n]}<br>{cor_titulo}: {cor_por[n]}" for n in nos],
        hoverinfo="text", name="páginas"))
    fig = go.Figure(traces)
    fig.update_layout(title=titulo, showlegend=False, width=largura, height=altura, plot_bgcolor="white",
                      xaxis=dict(visible=False), yaxis=dict(visible=False), margin=dict(l=10, r=10, t=50, b=10),
                      dragmode="pan")
    return fig

indeg = dict(gsub.in_degree())
TOP = 150
top_nodes = [n for n, _ in sorted(indeg.items(), key=itemgetter(1), reverse=True)[:TOP]]
gtop = gsub.subgraph(top_nodes)
pos_top = nx.spring_layout(gtop, k=0.35, seed=42)
camada_top = {n: gtop.nodes[n].get("layer", 2) for n in gtop}

desenhar_rede(gtop, pos_top, tamanho=indeg, cor_por=camada_top,
              titulo=f"Os {TOP} assuntos mais citados (tamanho = indegree, cor = camada)").show()

Agora o ranking explícito. `gsub.in_degree()` e `gsub.out_degree()` funcionam como `degree()`, mas contam só as arestas que entram ou só as que saem.

In [ ]:
top_in = sorted(indeg.items(), key=itemgetter(1), reverse=True)[:30]
fig = go.Figure(go.Bar(x=[v for _, v in top_in][::-1], y=[n for n, _ in top_in][::-1], orientation="h",
                       marker_color=AZUL, text=[v for _, v in top_in][::-1], textposition="outside"))
fig.update_layout(title="Top 30 por grau de entrada (quantas páginas apontam para ela)",
                  width=750, height=750, plot_bgcolor="white", xaxis_title="indegree",
                  margin=dict(l=10))
fig.show()

🧠 **Pare e pense.** Muitos nós do topo têm indegree parecido, na casa de 40 a 60. Isso é coincidência? *(Dica: quantos nós há na camada 1? Quem é capaz de apontar para um nó, dado que só baixamos as camadas 0 e 1?)* Este é um exemplo clássico de **artefato de amostragem**: o ranking diz tanto sobre *como* coletamos quanto sobre a Wikipédia.

### 6.1 Entrada versus saída

Uma página muito citada é também uma página que cita muito? Cada ponto abaixo é uma página; passe o mouse para identificá-la.

In [ ]:
outdeg = dict(gsub.out_degree())
nos = list(gsub)
fig = go.Figure(go.Scatter(
    x=[outdeg[n] for n in nos], y=[indeg[n] for n in nos], mode="markers",
    marker=dict(color=AZUL, size=7, opacity=0.55, line=dict(color="white", width=0.5)),
    hovertext=nos, hoverinfo="text"))
fig.update_layout(title="Cada ponto é uma página: quanto cita (x) versus quanto é citada (y)",
                  xaxis_title="outdegree (links que saem)", yaxis_title="indegree (links que chegam)",
                  width=700, height=500, plot_bgcolor="white")
fig.update_xaxes(gridcolor="#eee"); fig.update_yaxes(gridcolor="#eee")
fig.show()

print("Páginas com outdegree 0 (nunca baixadas):", sum(1 for n in nos if outdeg[n] == 0))

Observe a nuvem no eixo vertical à direita: páginas da camada 1 têm outdegree alto (baixamos o conteúdo delas) e páginas da camada 2 têm outdegree **zero** (nunca baixamos). A rede que temos não é "a Wikipédia perto de Complex network"; é **a Wikipédia vista da janela de Complex network**. Toda rede coletada por snowball carrega esse viés. Saber disso é metade da competência de um analista de redes.